TPU backend validation only. No competition data or model-quality claim.

In [ ]:
from pathlib import Path
import subprocess,sys
worker=Path("/kaggle/working/tpu_backend_probe.py")
worker.write_text('"""Short backend validation only: no competition images, labels or predictions."""\nimport importlib.metadata as metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nos.environ.setdefault("PJRT_DEVICE", "TPU")\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\nOUT = Path("/kaggle/working/tpu_backend_check")\nOUT.mkdir(exist_ok=True)\n\n\ndef worker(index):\n    import torch\n    import torch_xla\n    import torch_xla.core.xla_model as xm\n    import torch_xla.runtime as xr\n    from torch import nn\n    from torch.nn import functional as F\n\n    started = time.time()\n    torch.set_num_threads(1)\n    torch.manual_seed(20261109)\n    device = torch_xla.device()\n    rank, world = xr.global_ordinal(), xr.world_size()\n    if xr.device_type() != "TPU" or world != 8:\n        raise RuntimeError(f"Expected eight TPU workers, got {xr.device_type()} / {world}")\n    model = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.GroupNorm(4, 16), nn.GELU(),\n                          nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.GroupNorm(8, 32), nn.GELU(),\n                          nn.Conv2d(32, 16, 3, padding=1), nn.GELU(),\n                          nn.Upsample(size=(256, 256), mode="bilinear", align_corners=False),\n                          nn.Conv2d(16, 1, 1)).to(device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=.01)\n    generator = torch.Generator().manual_seed(400 + rank)\n    image = torch.randn((2, 3, 256, 256), generator=generator).to(device)\n    target = torch.zeros((2, 1, 256, 256))\n    target[:, :, 120 + rank:128 + rank, 30:220] = 1\n    target = target.to(device)\n    losses = []\n    for step in range(4):\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast("xla", dtype=torch.bfloat16):\n            logits = model(image)\n        logits = logits.float()\n        probability = logits.sigmoid()\n        loss = F.binary_cross_entropy_with_logits(logits, target)\n        loss = loss + (1 - (2 * (probability * target).sum() + 1) /\n                       (probability.sum() + target.sum() + 1))\n        loss.backward()\n        xm.optimizer_step(optimizer)\n        torch_xla.sync()\n        losses.append(float(loss.cpu()))\n        if not math.isfinite(losses[-1]):\n            raise FloatingPointError("Non-finite TPU loss")\n    parameters = torch.cat([p.detach().cpu().flatten() for p in model.parameters()])\n    gradients = torch.cat([p.grad.detach().cpu().flatten() for p in model.parameters()])\n    if not torch.isfinite(parameters).all() or not torch.isfinite(gradients).all():\n        raise FloatingPointError("Non-finite TPU parameters or gradients")\n    row = dict(rank=rank, world=world, device_type=xr.device_type(), losses=losses,\n               parameters=float(parameters.double().sum()),\n               parameter_square_sum=float(parameters.double().square().sum()),\n               finite_gradients=True, seconds=time.time()-started, precision="bfloat16 convolution / float32 loss")\n    (OUT / f"rank_{rank}.json").write_text(json.dumps(row, indent=2))\n    print(json.dumps(row), flush=True)\n\n\nif __name__ == "__main__":\n    started = time.time()\n    versions = {}\n    for package in ("torch", "torch-xla", "torchvision", "libtpu", "torch-tpu", "jax"):\n        try:\n            versions[package] = metadata.version(package)\n        except metadata.PackageNotFoundError:\n            versions[package] = None\n    inventory = dict(packages=versions, cpu_count=os.cpu_count(),\n                     memory=Path("/proc/meminfo").read_text().splitlines()[:2],\n                     competition_data_used=False, model_quality_measured=False)\n    (OUT / "inventory.json").write_text(json.dumps(inventory, indent=2))\n    print(json.dumps(inventory), flush=True)\n    import torch_xla\n    torch_xla.launch(worker, args=())\n    rows = [json.loads((OUT / f"rank_{rank}.json").read_text()) for rank in range(8)]\n    for key in ("parameters", "parameter_square_sum"):\n        values = [row[key] for row in rows]\n        if max(values)-min(values) > 1e-5:\n            raise RuntimeError("TPU gradient reduction did not synchronize parameters")\n    result = dict(seconds=time.time()-started, workers=8, parameter_synchronization_verified=True,\n                  finite_gradients_all_workers=True, model_quality_measured=False)\n    (OUT / "result.json").write_text(json.dumps(result, indent=2))\n    print("TPU backend validation complete:", json.dumps(result), flush=True)\n',encoding="utf8")
subprocess.run([sys.executable,str(worker)],check=True,timeout=480)